In [ ]:
import os
import torch
import torchvision
import torchvision.transforms as T
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torch.utils.data import DataLoader
from PIL import Image
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
import pandas as pd
import csv
from tqdm import tqdm
import xml.etree.ElementTree as ET
from torchvision.ops import box_iou
import matplotlib.pyplot as plt

# Configuration
visual_debug = False  # Set True to visualize predictions
score_threshold = 0.5
iou_threshold = 0.5
num_classes = 3  # 0 is background

# Dataset
class VOCLikeDataset(torch.utils.data.Dataset):
    def __init__(self, images_dir, labels_dir, transforms=None):
        self.images_dir = images_dir
        self.labels_dir = labels_dir
        self.transforms = transforms
        self.image_files = sorted([f for f in os.listdir(images_dir) if f.endswith((".jpg", ".png"))])
        self.class_map = {
            "fall": 1,
            "no_fall": 2,
        }

    def __getitem__(self, idx):
        img_file = self.image_files[idx]
        img_path = os.path.join(self.images_dir, img_file)
        label_file = img_file.replace(".jpg", ".xml").replace(".png", ".xml")
        label_path = os.path.join(self.labels_dir, label_file)

        img = Image.open(img_path).convert("RGB")
        tree = ET.parse(label_path)
        root = tree.getroot()

        boxes, labels = [], []
        for obj in root.findall("object"):
            label = obj.find("name").text
            if label not in self.class_map:
                continue
            labels.append(self.class_map[label])
            bbox = obj.find("bndbox")
            x_min = int(float(bbox.find("xmin").text))
            y_min = int(float(bbox.find("ymin").text))
            x_max = int(float(bbox.find("xmax").text))
            y_max = int(float(bbox.find("ymax").text))
            boxes.append([x_min, y_min, x_max, y_max])

        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)
        target = {"boxes": boxes, "labels": labels}

        if self.transforms:
            img = self.transforms(img)

        return img, target

    def __len__(self):
        return len(self.image_files)

# Transforms
def get_transform():
    return T.Compose([
        T.ToTensor()
    ])

# Matching predictions to ground truths
def match_predictions(pred_boxes, pred_labels, gt_boxes, gt_labels, iou_thresh=0.5):
    matches = []
    ious = box_iou(pred_boxes, gt_boxes)
    gt_used = set()

    for i in range(len(pred_boxes)):
        max_iou, gt_idx = ious[i].max(0)
        if max_iou >= iou_thresh and gt_idx.item() not in gt_used:
            matches.append((pred_labels[i].item(), gt_labels[gt_idx].item()))
            gt_used.add(gt_idx.item())
        else:
            matches.append((pred_labels[i].item(), 0))  # unmatched pred = FP

    for i in range(len(gt_boxes)):
        if i not in gt_used:
            matches.append((0, gt_labels[i].item()))  # missed GT = FN

    return matches

# Evaluation
def evaluate_model(model, dataloader, device, epoch=None, set_name="val", save_dir="runs/faster_rcnn"):
    model.eval()
    all_preds, all_targets = [], []

    with torch.no_grad():
        for images, targets in tqdm(dataloader, desc=f"Evaluating {set_name}"):
            images = [img.to(device) for img in images]
            outputs = model(images)

            for img, output, target in zip(images, outputs, targets):
                pred_boxes = output["boxes"].cpu()
                pred_labels = output["labels"].cpu()
                scores = output["scores"].cpu()

                gt_boxes = target["boxes"].cpu()
                gt_labels = target["labels"].cpu()

                keep = scores > score_threshold
                pred_boxes = pred_boxes[keep]
                pred_labels = pred_labels[keep]

                matches = match_predictions(pred_boxes, pred_labels, gt_boxes, gt_labels)

                for pred, gt in matches:
                    all_preds.append(pred)
                    all_targets.append(gt)

                if visual_debug and epoch == 1:
                    visualize(img.cpu(), pred_boxes, gt_boxes)

    labels_range = list(range(1, num_classes))  # exclude 0
    precision = precision_score(all_targets, all_preds, labels=labels_range, average="weighted", zero_division=0)
    recall = recall_score(all_targets, all_preds, labels=labels_range, average="weighted", zero_division=0)
    f1 = f1_score(all_targets, all_preds, labels=labels_range, average="weighted", zero_division=0)
    cm = confusion_matrix(all_targets, all_preds, labels=list(range(num_classes)))

    print(f"{set_name.upper()} Epoch {epoch} — Precision: {precision:.4f}, Recall: {recall:.4f}, F1 Score: {f1:.4f}")

    os.makedirs(save_dir, exist_ok=True)
    with open(f"{save_dir}/results_{set_name}.csv", mode="a", newline="") as file:
        writer = csv.writer(file)
        if epoch == 1:
            writer.writerow(["Epoch", "Precision", "Recall", "F1 Score"])
        writer.writerow([epoch, precision, recall, f1])

    with open(f"{save_dir}/conf_matrix_{set_name}.csv", mode="a", newline="") as f:
        f.write(f"Confusion Matrix - Epoch {epoch}\n")
        pd.DataFrame(cm).to_csv(f, header=False, index=False)
        f.write("\n")

# Training
def train_one_epoch(model, dataloader, optimizer, device):
    model.train()
    total_loss = 0.0
    for images, targets in dataloader:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())
        total_loss += losses.item()
        optimizer.zero_grad()
        losses.backward()
        optimizer.step()
    return total_loss / len(dataloader)

# Visualization
def visualize(img, pred_boxes, gt_boxes):
    import matplotlib.patches as patches
    img = img.permute(1, 2, 0).numpy()
    fig, ax = plt.subplots(1, figsize=(10, 10))
    ax.imshow(img)
    for box in gt_boxes:
        x1, y1, x2, y2 = box.tolist()
        ax.add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1, edgecolor='g', facecolor='none', linewidth=2, label='GT'))
    for box in pred_boxes:
        x1, y1, x2, y2 = box.tolist()
        ax.add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1, edgecolor='r', facecolor='none', linewidth=2, label='Pred'))
    plt.show()

# Main
def main():
    train_img_dir = "datasets_paper/images/train"
    train_label_dir = "datasets_paper/labels_voc/train"
    val_img_dir = "datasets_paper/images/val"
    val_label_dir = "datasets_paper/labels_voc/val"

    lr = 0.005
    num_epochs = 50
    batch_size = 4
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    train_dataset = VOCLikeDataset(train_img_dir, train_label_dir, transforms=get_transform())
    val_dataset = VOCLikeDataset(val_img_dir, val_label_dir, transforms=get_transform())

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))
    val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, collate_fn=lambda x: tuple(zip(*x)))

    model = fasterrcnn_resnet50_fpn(weights="DEFAULT")
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = torchvision.models.detection.faster_rcnn.FastRCNNPredictor(in_features, num_classes)
    model.to(device)

    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=0.0005)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

    for epoch in range(1, num_epochs + 1):
        print(f"\nEpoch {epoch}/{num_epochs}")
        train_loss = train_one_epoch(model, train_loader, optimizer, device)
        print(f"Train Loss: {train_loss:.4f}")

        evaluate_model(model, train_loader, device, epoch=epoch, set_name="train")
        evaluate_model(model, val_loader, device, epoch=epoch, set_name="val")

        scheduler.step()

    torch.save(model.state_dict(), f"runs/faster_rcnn/final_model.pt")
    print("Model saved.")

if __name__ == "__main__":
    main()
